In [4]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Notebook-safe repo root detection
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
ROOT = next((p for p in candidates if (p / "CDRPS").exists()), Path.cwd())

DATA_PATH = ROOT / "CDRPS" / "data" / "processed" / "df_scaled.csv"
MODELS_DIR = ROOT / "CDRPS" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Using data:", DATA_PATH)
print("Saving artifacts to:", MODELS_DIR)

df = pd.read_csv(DATA_PATH)

target_col = "Delay_Risk_Index"
numeric_df = df.select_dtypes(include=["int64", "float64"]).copy()

if target_col not in numeric_df.columns:
    feature_pool = [
        c
        for c in numeric_df.columns
        if c != "Respondent Number" and not c.startswith("Respondant Information")
    ]
    if not feature_pool:
        raise ValueError(
            "Could not auto-create Delay_Risk_Index because no risk-factor numeric columns were found."
        )
    numeric_df[target_col] = numeric_df[feature_pool].mean(axis=1)
    print(
        f"{target_col} not found. Auto-generated from {len(feature_pool)} risk-factor columns."
    )

X_df = numeric_df.drop(columns=[target_col])
y = numeric_df[target_col].values

feature_columns = list(X_df.columns)
X = X_df.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

joblib.dump(model, MODELS_DIR / "model.pkl")
joblib.dump(scaler, MODELS_DIR / "scaler.pkl")

with open(MODELS_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=2)

print("Artifacts saved to:", MODELS_DIR)

Using data: c:\Users\olive\Desktop\My ML Material\CONSTRUCTION DELAY RISK PREDICTION SYSTEM\CDRPS\data\processed\df_scaled.csv
Saving artifacts to: c:\Users\olive\Desktop\My ML Material\CONSTRUCTION DELAY RISK PREDICTION SYSTEM\CDRPS\models
Delay_Risk_Index not found. Auto-generated from 44 risk-factor columns.
MAE: 0.1145974581419655
RMSE: 0.15917874627885684
R²: 0.9401205916973085
Artifacts saved to: c:\Users\olive\Desktop\My ML Material\CONSTRUCTION DELAY RISK PREDICTION SYSTEM\CDRPS\models
